# HouseGen cloud studio (Kaggle, free GPU)

Makes every song waiting in the site's queue on Kaggle's free GPU, then stops.

**One-time setup**
1. Settings (right panel) → **Accelerator: GPU T4 x2** and **Internet: On** (needs a phone-verified account).
2. Add-ons → **Secrets**: add `HOUSEGEN_API_URL` and `HOUSEGEN_WORKER_TOKEN` (same values as your `.env`),
   plus `GROQ_API_KEY` for lyrics. Optional: `GROQ_MODEL`, `SPOTIPY_CLIENT_ID`, `SPOTIPY_CLIENT_SECRET`,
   `ELEVENLABS_API_KEY`. Tick each secret so this notebook can read it.

**Each time songs are waiting**: click **Save Version → Save & Run All**. It runs in the background, so you can
close the tab. The first song takes longer while ACE-Step installs and downloads its models (~10 GB).

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

REQUIRED_SECRETS = ["HOUSEGEN_API_URL", "HOUSEGEN_WORKER_TOKEN"]
OPTIONAL_SECRETS = ["GROQ_API_KEY", "GROQ_MODEL", "SPOTIPY_CLIENT_ID", "SPOTIPY_CLIENT_SECRET", "ELEVENLABS_API_KEY"]

_client = UserSecretsClient()


def load_secret(name: str) -> bool:
    """Copy a Kaggle secret into the environment; False if it isn't added or ticked for this notebook."""
    try:
        os.environ[name] = _client.get_secret(name)
        return True
    except Exception:
        return False


missing = [name for name in REQUIRED_SECRETS if not load_secret(name)]
if missing:
    raise RuntimeError(f"Add these Kaggle secrets (Add-ons -> Secrets) and tick them for this notebook: {missing}")
loaded = [name for name in OPTIONAL_SECRETS if load_secret(name)]
print("secrets ok; optional loaded:", loaded or "none")
if "GROQ_API_KEY" not in loaded:
    print("warning: no GROQ_API_KEY, so lyric themes fall back to a public-domain poem")

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU: Settings -> Accelerator -> GPU T4 x2")
print(gpu.stdout.strip())
if "P100" in gpu.stdout:
    raise RuntimeError("ACE-Step's PyTorch build doesn't support the P100; switch to GPU T4 x2")

In [ ]:
%%bash
# Same ACE-Step commit as the Mac install, so the Take 4 sound carries over.
set -euo pipefail
ACE_COMMIT=ca1e85fe9430179831e6bc6be790c332190a3866
cd /tmp
rm -rf housegen ace
git clone --quiet --depth 1 https://github.com/mehulvar/HouseMusicGenerator.git housegen
git clone --quiet https://github.com/ACE-Step/ACE-Step-1.5.git ace
git -C ace checkout --quiet "$ACE_COMMIT"
pip install -q uv
(cd ace && uv sync --frozen --no-dev --python 3.11)
pip install -q -r housegen/requirements.txt
command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg)
echo "install ok"

In [ ]:
import os
import subprocess
import sys
import time

ACE_DIR = "/tmp/ace"
ACE_PORT = 8001
ACE_BOOT_TIMEOUT_S = 25 * 60  # first boot downloads the models
ACE_LOG = "/tmp/acestep.log"
LOG_TAIL_LINES = 40

# Match the Mac studio: regular turbo DiT (not XL) and the 1.7B planner LM, thinking on.
os.environ.update({
    "ACESTEP_CONFIG_PATH": "acestep-v15-turbo",
    "ACESTEP_LM_MODEL_PATH": "acestep-5Hz-lm-1.7B",
    "ACESTEP_LM_BACKEND": "pt",
    "ACESTEP_URL": f"http://127.0.0.1:{ACE_PORT}",
    "TOKENIZERS_PARALLELISM": "false",
})
sys.path.insert(0, "/tmp/housegen")
import worker  # noqa: E402  (reads HOUSEGEN_* from the environment set above)
from housegen import audio_acestep  # noqa: E402


def print_log_tail():
    with open(ACE_LOG) as f:
        print("".join(f.readlines()[-LOG_TAIL_LINES:]))


def wait_for_acestep(server: subprocess.Popen):
    deadline = time.time() + ACE_BOOT_TIMEOUT_S
    while not audio_acestep.available():
        if server.poll() is not None:
            print_log_tail()
            raise RuntimeError(f"ACE-Step exited during startup (code {server.returncode})")
        if time.time() > deadline:
            print_log_tail()
            raise TimeoutError(f"ACE-Step not healthy after {ACE_BOOT_TIMEOUT_S}s")
        time.sleep(5)
    worker.log("ACE-Step is up")


with worker.Heartbeat(), open(ACE_LOG, "w") as ace_log:  # site shows a studio online while this runs
    server = subprocess.Popen(
        [f"{ACE_DIR}/.venv/bin/acestep-api", "--host", "127.0.0.1", "--port", str(ACE_PORT)],
        cwd=ACE_DIR, stdout=ace_log, stderr=subprocess.STDOUT,
    )
    try:
        wait_for_acestep(server)
        worker.log(f"made {worker.drain()} song(s)")
    finally:
        server.terminate()
        server.wait(timeout=60)